# IoT Traffic Classification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
df = pd.read_csv('IoT_dataset_with_flow_rates.csv')
print("Shape:", df.shape)
df.head()

## 2. EDA

In [ ]:
df['label'].value_counts().plot(kind='bar', figsize=(14,4), color='steelblue')
plt.title('Class Distribution')
plt.xlabel('Attack Type')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Rate asymmetry between source and destination
df['rate_ratio'] = df['Srate'] / (df['Drate'] + 1e-8)

# Total TCP flags active per flow
flag_cols = ['fin_flag_number','syn_flag_number','rst_flag_number','psh_flag_number','ack_flag_number']
df['total_flags'] = df[flag_cols].sum(axis=1)

# Average bytes per packet
df['bytes_per_pkt'] = df['Flow_Byts/s'] / (df['Flow_Pkts/s'] + 1e-8)

print("New features:", ['rate_ratio', 'total_flags', 'bytes_per_pkt'])
print("Shape:", df.shape)

## 4. Encode & Scale

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['label'])

X = df.drop(columns=['label']).replace([np.inf, -np.inf], 0)

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
print("X:", X_scaled.shape, "| Classes:", len(le.classes_))

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 6. Model

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score  : {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))